In [ ]:
# Gọi thêm các thư viện cần thiết cho mô hình mới
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

# Chia tách dữ liệu (giữ nguyên tỷ lệ 80-20 như cũ)
X = df_clean[['Area_Sqft', 'Baths']]
y = df_clean['Price_Rupees']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# 1. BIẾN ĐỔI LOGARIT (Log Transformation)
# Nén các mức giá trị khổng lồ xuống tỷ lệ nhỏ để AI dễ học hơn
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

# 2. HUẤN LUYỆN RANDOM FOREST
print("⏳ Đang gieo trồng 100 cây quyết định (Random Forest)...")
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train_log)

print("✅ Đã huấn luyện xong mô hình nâng cấp!")

In [ ]:
# 1. AI DỰ ĐOÁN (Kết quả lúc này đang ở dạng Logarit)
y_pred_log = rf_model.predict(X_test)

# 2. CHUYỂN ĐỔI NGƯỢC (Bung logarit ra lại thành tiền thật INR)
y_pred_rf = np.expm1(y_pred_log)

# 3. CHẤM ĐIỂM
r2_new = r2_score(y_test, y_pred_rf)
rmse_new = np.sqrt(mean_squared_error(y_test, y_pred_rf))

print("="*55)
print("🚀 BẢNG ĐIỂM CỦA MÔ HÌNH RANDOM FOREST:")
print(f"📈 Hệ số R-squared mới : {r2_new:.4f} (So với Linear Regression là 0.56)")
print(f"📉 Sai số RMSE mới     : {rmse_new:,.0f} INR")
print("="*55)

In [ ]:
# VẼ BIỂU ĐỒ SO SÁNH ĐỂ XEM ĐÁM MÂY DỮ LIỆU ĐÃ GỌN HƠN CHƯA
plt.figure(figsize=(8, 5))
plt.scatter(y_test, y_pred_rf, color='green', alpha=0.3, label='Random Forest Dự đoán')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Đường hoàn hảo (y=x)')

plt.xlabel('Giá Thực Tế (INR)')
plt.ylabel('Giá AI Dự Đoán (INR)')
plt.title('Random Forest + Log Transform: Thực tế vs Dự đoán')
plt.legend()
plt.grid(True)
plt.show()